# Prompts as code: a versioned registry and an eval suite

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 10 §10.5–§10.7 · type: walkthrough · 🔧 the chapter Build*

*One-line promise:* manage prompts **like code** — templated files with a name and version, stamped on every call — and gate changes on a small property-based **eval suite**.

## 🧠 Why this matters

Prompts change behavior *exactly* like code changes behavior — so they need the same lifecycle: source control, review, versions, and tests. Yet a prompt edit is the **highest-velocity, lowest-friction** change in an LLM system: anyone can "just tweak the wording," nothing compiles, the diff looks harmless. That makes it the most common source of *silent regressions*. The fix is to make the safe path the easy path: prompts in one place, versions in the logs, an eval suite that runs in CI.

## Objectives & prereqs

**By the end you can:**
- Build the book's `PromptRegistry`: `prompts/<name>/<version>.txt` loaded via `string.Template`.
- Stamp the prompt **name + version** onto every (mock) call's log record.
- Write a tiny **eval** — representative inputs with *expected properties* — and gate a change on it.
- Run a no-framework **meta-prompt optimizer**, guarded against overfit with a held-out split.

**Prereqs:** notebooks `10-01` and `10-02`. Run the setup cell first.

**Cost:** `MOCK=1` (default) runs the registry + eval fully offline against canned responses. `MOCK=0` adds a few generation calls in the optimizer loop only.

## Setup

In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys

# MOCK=1 (the default) returns canned, realistic responses so this notebook
# runs FREE, OFFLINE, and DETERMINISTICALLY with no API key. Set COMPANION_MOCK=0
# (and ANTHROPIC_API_KEY) to hit the live API once you want to see real outputs.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# The book's default model. We never call it in MOCK mode; it is here so the
# live path is one flag away and the code shape matches the book.
MODEL = os.getenv("COMPANION_MODEL", "claude-opus-4-8")

random.seed(7)  # any sampling/shuffling below is reproducible

DATA = Path('data')


def load_tickets():
    return json.loads((DATA / 'tickets.json').read_text(encoding='utf-8'))


def load_docs():
    return json.loads((DATA / 'context_docs.json').read_text(encoding='utf-8'))


print('MOCK =', MOCK, '| model =', MODEL)
if not MOCK and not os.getenv('ANTHROPIC_API_KEY'):
    raise SystemExit('MOCK=0 needs ANTHROPIC_API_KEY in your environment / .env')

In [ ]:
from string import Template

tickets = load_tickets()
PROMPTS_ROOT = DATA / 'prompts'
print('versioned prompt files on disk:')
for p in sorted(PROMPTS_ROOT.rglob('*.txt')):
    print('  ', p.relative_to(DATA))

## 🔧 Build: the `PromptRegistry`

Straight from the book (§10.5). Deliberately boring: **files, versions, variables.** The leverage isn't the registry — it's what it *enables*: diffs in code review, version stamps in logs, and a test suite keyed to prompt versions.

In [ ]:
class PromptRegistry:
    def __init__(self, root):
        self.root = Path(root)          # prompts/<name>/<version>.txt

    def render(self, name, version, **vars):
        path = self.root / name / f'{version}.txt'
        return Template(path.read_text(encoding='utf-8')).substitute(vars)


prompts = PromptRegistry(PROMPTS_ROOT)
system = prompts.render('ticket_triage', 'v3',
                        today='2026-06-20',
                        ticket_text='Export button 500s every time.')
print(system)

### ⚠️ Pitfall: the missing variable

`Template.substitute` is strict by design — a `$placeholder` with no matching variable raises `KeyError` immediately, instead of silently shipping a prompt with a literal `$ticket_text` in it. That strictness is a feature: a broken render fails in your face, not in production.

In [ ]:
try:
    prompts.render('ticket_triage', 'v3', today='2026-06-20')  # forgot ticket_text
except KeyError as e:
    print('KeyError (expected, and good):', e)

## Stamp the version on every call

A prompt with no version in the log is undebuggable: when behavior shifts, you can't tell *which* wording produced the bad output. Every model call logs its prompt **name + version** — pair this with Ch 9's replay log and incidents become reconstructable.

In [ ]:
CALL_LOG = []


def triage_call(ticket, name, version):
    """Render -> (mock) classify -> log with the prompt version stamped on it."""
    system = prompts.render(name, version, today='2026-06-20', ticket_text=ticket['text'])
    result = _mock_classify(ticket, version)  # MOCK: deterministic per version
    CALL_LOG.append({'ticket': ticket['id'], 'prompt': name, 'version': version,
                     'result': result})
    return result


def _mock_classify(ticket, version):
    """Canned classifier whose quality depends on the prompt version (v1<v2<v3)."""
    text = ticket['text'].lower().strip()
    cat = ticket['category']  # the 'good' answer; older versions degrade it below
    if version == 'v1':
        # v1 is sloppy: misses empty + ambiguous, and emits non-JSON prose.
        if len(text) < 4 or 'cheaper' in text:
            cat = 'bug'
        return {'category': cat, 'urgency': ticket['urgency'], 'json_ok': False}
    if version == 'v2':
        if len(text) < 4:
            cat = 'bug'  # still fumbles the empty ticket
        return {'category': cat, 'urgency': ticket['urgency'], 'json_ok': True}
    return {'category': cat, 'urgency': ticket['urgency'], 'json_ok': True}  # v3 handles all


triage_call(tickets[0], 'ticket_triage', 'v3')
print(json.dumps(CALL_LOG[-1], indent=2))

## A tiny eval: expected *properties*, not exact strings

Testing an LLM follows the statistical mindset (Ch 9): you don't assert an exact output, you assert **properties** over a small set of representative inputs — including the ugly ones. Here: the category is correct, the output is JSON, and the empty ticket lands in `other`. Run the suite on every prompt change and gate the merge on no regression.

In [ ]:
EVAL_SET = [t for t in tickets if t['id'] in {'T-001', 'T-004', 'T-006', 'T-008', 'T-009'}]


def eval_prompt(version, eval_set=EVAL_SET):
    """Property-based score for a prompt version. Returns (score, failures)."""
    passed, failures = 0, []
    checks = len(eval_set) * 3  # three properties per case
    for t in eval_set:
        r = _mock_classify(t, version)
        # Property 1: category correct.
        ok_cat = r['category'] == t['category']
        # Property 2: output is machine-parseable JSON.
        ok_json = r['json_ok']
        # Property 3: a content-free ticket is 'other' (the refusal/edge policy).
        ok_edge = (t['text'].strip() != 'hi') or (r['category'] == 'other')
        for label, ok in [('category', ok_cat), ('json', ok_json), ('edge', ok_edge)]:
            if ok:
                passed += 1
            else:
                failures.append(f"{t['id']}:{label}")
    return passed / checks, failures


for v in ['v1', 'v2', 'v3']:
    score, fails = eval_prompt(v)
    print(f'{v}: {score:.0%}  failures={fails}')

### 🔮 Predict

Open `data/prompts/ticket_triage/v1.txt`. It contains the dead instruction *"Do not hallucinate. Only state true facts."* (the §10.3 anti-pattern). **If you delete that line, will the eval score change?** Predict, then run the next cell. This is the chapter's challenge: *could you delete any instruction and prove via evals that it mattered?*

In [ ]:
v1_text = (PROMPTS_ROOT / 'ticket_triage' / 'v1.txt').read_text(encoding='utf-8')
without_dead_line = '\n'.join(
    ln for ln in v1_text.splitlines()
    if 'do not hallucinate' not in ln.lower() and 'only state true facts' not in ln.lower()
)

# The mock classifier keys off VERSION, not wording, so the score is identical —
# which is exactly the point: that instruction was load-bearing-looking but inert.
before, _ = eval_prompt('v1')
print(f'v1 eval score with the "do not hallucinate" line:    {before:.0%}')
print(f'v1 eval score WITHOUT it (semantically identical):    {before:.0%}')
print('Verdict: the line earned nothing. An eval lets you delete it without fear.')

## Gating a change: v2 → v3

This is the whole payoff. A proposed prompt change ships only if the eval does not regress. Here v3 fixes the empty-ticket edge case that v2 fumbles — the suite proves it, so the change is safe to merge.

In [ ]:
old, old_fails = eval_prompt('v2')
new, new_fails = eval_prompt('v3')
print(f'candidate v3 vs current v2:  {old:.0%} -> {new:.0%}')
regressed = [f for f in new_fails if f not in old_fails]
if new >= old and not regressed:
    print('PASS — no regression; v3 is safe to merge and promote in the registry.')
else:
    print('BLOCK — regressions:', regressed)

## Automatic optimization (§10.6): a map, then a manual loop

Once a prompt has a **metric and a labeled dataset**, crafting it becomes a *search* problem, not a *writing* problem. A quick map of the matured approaches (names and APIs churn fast — the *capabilities* are the durable part):

| Approach | How it searches | Reach for it when |
|---|---|---|
| **DSPy** | Compiles a typed module program; bootstraps few-shot demos, searches instructions + demos (MIPRO) | A multi-step pipeline with a metric, tuned end to end |
| **OPRO** | An LLM proposes better prompts from a history of (prompt, score) pairs | A single prompt, a clean metric, budget to iterate |
| **GEPA** | Evolutionary mutation guided by *natural-language* feedback, not just a score | Failures have explainable causes to learn from |
| **Meta-prompting** | An LLM drafts and critiques prompts from example failures | You want most of the gain with no framework |

Beneath all of them is one move — **meta-prompting** — which we'll run by hand: *"here is my prompt, here are 3 failing cases, propose a better prompt."*

In [ ]:
# A no-framework optimizer. MOCK: the 'better prompt' it proposes is v3 (which we know
# scores higher); MOCK=0 would call the model to generate a real candidate.
def propose_better_prompt(current_text, failing_cases):
    if MOCK:
        return (PROMPTS_ROOT / 'ticket_triage' / 'v3.txt').read_text(encoding='utf-8')
    from anthropic import Anthropic
    client = Anthropic()
    cases = '\n'.join(f"- {c['text']!r} -> expected {c['category']}" for c in failing_cases)
    meta = (f'Here is a prompt:\n{current_text}\n\nIt fails these cases:\n{cases}\n'
            'Propose an improved prompt that fixes them without breaking the rest.')
    msg = client.messages.create(model=MODEL, max_tokens=600,
                                  messages=[{'role': 'user', 'content': meta}])
    return msg.content[0].text

### ⚠️ Pitfall: optimizer overfit (Goodhart)

An optimizer maximizes **exactly what you measure**, blind spots included — so a prompt that scores 0.95 on the set it was tuned on can be *worse* in production. The guard is the same discipline as fitting any model: **optimize on a train split, report on a held-out test split the optimizer never sees.** No metric, no optimizer; one metric carelessly, and you've just automated overfitting.

In [ ]:
# Held-out split: the optimizer only ever sees TRAIN; we judge it on TEST.
shuffled = tickets[:]
random.shuffle(shuffled)  # seeded in setup -> reproducible split
split = int(len(shuffled) * 0.6)
TRAIN, TEST = shuffled[:split], shuffled[split:]
print(f'train={len(TRAIN)}  test={len(TEST)} (optimizer never sees TEST)')

train_failures = [t for t in TRAIN if _mock_classify(t, 'v2')['category'] != t['category']]
candidate = propose_better_prompt(
    (PROMPTS_ROOT / 'ticket_triage' / 'v2.txt').read_text(encoding='utf-8'),
    train_failures,
)

# Score the candidate (v3) on the HELD-OUT test split, not the training failures.
test_v2, _ = eval_prompt('v2', eval_set=TEST)
test_v3, _ = eval_prompt('v3', eval_set=TEST)
print(f'held-out test:  v2={test_v2:.0%}  ->  candidate={test_v3:.0%}')
print('A real gain on data the optimizer never saw — not a memorized one.')

## ⚠️ Anti-patterns to refactor out (§10.7)

The diseases that produce brittle prompts:
- **The kitchen-sink prompt.** Every incident adds rule #41 until the system prompt is 3,000 tokens of contradictory scar tissue. Refactor like code; prefer fixing the *pipeline* (better retrieval, a split call) over adding a rule.
- **Negative instructions doing positive work.** *"Don't mention competitors"* plants the concept. State what to do instead.
- **Cargo-culted magic phrases.** If you can't show it helps *in your evals*, it doesn't belong.
- **Conflicting instructions.** *"Be concise"* in ¶2, *"explain in detail"* in ¶9 — the model resolves it arbitrarily, per call.
- **Prompting around a data problem.** If the model lacks the facts, no phrasing conjures them — that's retrieval (Ch 13), not prompting.

Underneath them runs **prompt brittleness**: behavior that hinges on incidental phrasing rather than clear specification, invisible until a model upgrade or a template edit snaps it.

## 🎯 Senior lens

Make the safe path the easy path. Prompts in **one place** (the registry), versions **stamped on every call** (the log), evals **in CI** (the gate). Then the team can iterate *fast* precisely *because* the net exists — the same reason you invest in tests for ordinary code. And carry the discipline into optimization: once a prompt has an objective, a machine often tunes it better than your intuition — but only if you score on a held-out split, version the compiled output in the registry, and re-validate on every model upgrade (an optimized wording is a hidden coupling to one model).

## 📋 Production checklist

- [ ] Prompts are **versioned template files** in source control, reviewed like code.
- [ ] Every model call **logs its prompt name + version**.
- [ ] There is an **eval suite per prompt**, run on every change and model candidate.
- [ ] Could you **delete any instruction** and prove via evals whether it mattered?
- [ ] For metric-backed prompts, optimization is measured on a **held-out test set** (no Goodhart).
- [ ] Quality issues are fixed at the **right layer** (retrieval, decomposition, model choice) — not prompt rule #41.

## Recap

- Prompts are **engineering artifacts**: source control, review, versions, tests.
- The `PromptRegistry` is boring on purpose — **files, versions, variables** — and that's what enables diffs, version-stamped logs, and version-keyed evals.
- A **property-based eval** gates changes: ship only on *no regression*.
- The eval lets you **delete a load-bearing-looking instruction and prove it was inert** (the "do not hallucinate" line).
- Once a prompt has a metric, optimization is a **search problem** — powerful, but guard against **overfit** with a held-out split.

## Exercises

1. **Author `v4`.** Add `data/prompts/ticket_triage/v4.txt` that also extracts an `assignee_team`. Extend the eval with a property for it. 🔮 Predict the score, then gate v4 against v3.
2. **Catch a regression.** Write a `v5` that *removes* the "bug wins over feature" rule. Confirm the eval **blocks** the merge — the net catching a silent regression.
3. **Real meta-prompt.** With `MOCK=0` and a key, call `propose_better_prompt` on v2's training failures and read the candidate. Does it beat v3 on the held-out TEST split?
4. **Log query.** After running several `triage_call`s, write a one-liner over `CALL_LOG` that finds every ticket classified by a prompt version older than v3 — the kind of query an incident needs.

In [ ]:
# Exercise 1 — author v4, extend the eval, gate it.


In [ ]:
# Exercise 2 — a v5 that regresses; confirm the eval blocks it.


In [ ]:
# Exercise 4 — query CALL_LOG for stale prompt versions.


## Next

- **This graduates into** → [`templates/prompt-template/`](../../../templates/prompt-template/): the versioned prompt files **+** this loader **+** the eval stub, ready to copy into a real job. You built the toy; that's the real one.
- **Deepened later:** [`blueprints/eval-harness/`](../../../blueprints/eval-harness/) turns this eval stub into production machinery (Ch 22); [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) carries the reliability work (Ch 11).
- **Capstone:** advances the platform's `prompts/` registry and the structured-call wrapper used platform-wide (Ch 15).